# satreg on a Kaggle GPU

Runs `satreg benchmark` on this notebook's GPU and saves the run's directory
(`summary.md`, `summary.csv`, `pairs.csv`, `context.json`) as the notebook's
output, under `outputs/<config name>/`. The procedure, from pushing the notebook
to merging its results with a run on another machine, is in `docs/kaggle.md` in
the repository.

Settings: accelerator **GPU T4 x2** (the benchmark uses one), internet **on**.
While the repository is private, attach a Kaggle secret named `GITHUB_TOKEN`
holding a read-only GitHub token for it.

In [ ]:
# What to run. The commit must be pushed to GitHub; context.json records the
# one that ran, so a row can be traced to its code.
REPOSITORY = "https://github.com/esamoun/ai-satellite-image-registration.git"
COMMIT = "main"
# A run config of the repository; every deep-learning method it lists runs on
# the GPU, the others on the CPU.
CONFIG = "configs/gpu.toml"
# More options for `satreg benchmark`, which override the config's, e.g.
# ["--pairs", "2"] for a quick check.
OPTIONS = []
# RoMa needs its own install and 1.7 GB of weights; leave it out if the config
# does not list it.
WITH_ROMA = True

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Kaggle saves /kaggle/working as the notebook's output; the code and the chips
# stay outside it.
WORKING = Path("/kaggle/working")
CODE = Path("/tmp/satreg")


def run(*command, cwd=None, env=None):
    """Run a command, print its output, stop the notebook if it fails."""
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, env=env, check=True)


def timed(step, started):
    print(f"{step}: {time.perf_counter() - started:.0f} s", flush=True)

In [ ]:
run("nvidia-smi")

In [ ]:
# Clone, with the token of the GITHUB_TOKEN secret if one is attached. The token
# goes to git through the environment only: it is in no command line, in no
# file of the clone, and in no output.
started = time.perf_counter()
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:  # no secret attached, or not on Kaggle: a public repository
    token = None

git, env = ["git"], dict(os.environ)
if token:
    env["GITHUB_TOKEN"] = token
    helper = '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f'
    # An empty helper first drops any helper the machine configured, which
    # git would otherwise ask before this one.
    git += ["-c", "credential.helper=", "-c", f"credential.helper={helper}"]
shutil.rmtree(CODE, ignore_errors=True)
run(*git, "clone", "--quiet", REPOSITORY, CODE, env=env)
del token, env
run("git", "checkout", "--quiet", COMMIT, cwd=CODE)
run("git", "log", "-1", "--format=%H %s", cwd=CODE)
timed("clone", started)

In [ ]:
# The dl extra (and RoMa), then the pretrained weights, downloaded once per
# session to PyTorch's cache, ~/.cache/torch, outside the saved output. The
# benchmark's warm-up call would download them too, untimed; fetching them here
# keeps the download apart from the run.
started = time.perf_counter()
run("make", "install-roma" if WITH_ROMA else "install-dl", cwd=CODE)
timed("install and weights", started)

In [ ]:
started = time.perf_counter()
out = WORKING / "outputs" / Path(CONFIG).stem
command = ["benchmark", "--config", CONFIG, "--device", "cuda", "--out", out]
run(sys.executable, "-m", "satreg", *command, *OPTIONS, cwd=CODE)
timed("benchmark", started)

In [ ]:
context = json.loads((out / "context.json").read_text())
print(json.dumps({key: context[key] for key in ("code", "gpu", "devices")}, indent=2))
print(f"run: {context['cost']['run_seconds'] / 60:.1f} min")